# Compare saved models in Jupyter

Load each past run's checkpoint and measure something new on one shared evaluation batch. We'll first inspect the recorded runs and saved checkpoints, then rebuild the models and compare them.

Launch Jupyter inside the experiment's Git repository, where FlorDB finds its `.flor` database and object store. Run the next cell to see what FlorDB recorded; no project settings are needed yet. The worked example uses `examples/v4/train.py`, and the later cells explain what to change for your project.

In [ ]:
import pandas as pd

import flordb as flor

recorded_runs = flor.dataframe()
if recorded_runs.empty:
    raise RuntimeError("No FlorDB runs found. Launch Jupyter inside the experiment's Git repository.")

recorded_runs

Choose a training script from the `filename` column above. We'll compare its original (`forward`) runs. The table below shows the arguments and metrics recorded for that script; the MNIST example records `hidden`, which determines the model's hidden layer size.

In [ ]:
SCRIPT = "train.py"

forward = recorded_runs[recorded_runs.source.eq("forward")]
scripts = sorted(forward.filename.unique())
if SCRIPT not in scripts:
    raise ValueError(f"No runs of {SCRIPT!r}. Set SCRIPT to one of: {scripts}")

script_runs = forward[forward.filename.eq(SCRIPT)].dropna(axis="columns", how="all")
script_runs

Now inspect the local checkpoints for these runs. Entries with `kind == "run"` are files saved by the training script with `torch.save`; their `name` is the path the script used. The MNIST example saves `ckpt.pth`.

In [ ]:
saved = {t: flor.checkpoints(t) for t in script_runs.tstamp.unique()}
pd.concat(
    [checkpoints.assign(tstamp=t) for t, checkpoints in saved.items()],
    ignore_index=True,
)

Use the two tables above to configure the comparison. Set `ARCH_ARGS` to the recorded `flor.arg` names needed to rebuild your model, and `CHECKPOINT` to the saved file to load from each run. The defaults use the MNIST example's `hidden` argument and `ckpt.pth` checkpoint.

The next cell keeps one row per run with the required arguments and a local checkpoint. Then we'll define how to rebuild and evaluate those models.

In [ ]:
ARCH_ARGS = ["hidden"]
CHECKPOINT = "ckpt.pth"

runs = script_runs.copy()
recorded = [c for c in runs.columns if c not in ("projid", "tstamp", "filename", "source")]
missing = [a for a in ARCH_ARGS if a not in recorded]
if missing:
    raise ValueError(f"{SCRIPT} recorded no {missing}. Set ARCH_ARGS from: {recorded}")
runs = runs.dropna(subset=ARCH_ARGS).drop_duplicates("tstamp")

has_checkpoint = runs.tstamp.map(lambda t: CHECKPOINT in set(saved[t].name))
if not has_checkpoint.any():
    names = sorted({n for c in saved.values() for n in c.name[c.kind.eq("run")]})
    raise ValueError(
        f"No local checkpoint named {CHECKPOINT!r}. Set CHECKPOINT to one of: {names}"
        if names else
        "No local run checkpoints. Checkpoints don't travel with Git: copy .flor/obj_store from the training machine."
    )
if not has_checkpoint.all():
    print(f"Skipping {(~has_checkpoint).sum()} run(s) without a local {CHECKPOINT!r}.")
runs = runs[has_checkpoint].copy()
runs

**Edit for your project.** `make_model(row)` rebuilds the model a run trained, from that run's recorded arguments. If the architecture changed between runs, pick the right one here. `state_dict_of` pulls the model's state out of what the script saved. This pair matches `examples/v4/train.py`.

In [ ]:
import torch
from torch import nn


class NeuralNet(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(784, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 10)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


def make_model(row):
    return NeuralNet(int(row["hidden"]))


def state_dict_of(checkpoint):
    # train.py saves {"model": ..., "optimizer": ...}.
    # For torch.save(model.state_dict(), ...), return checkpoint itself.
    return checkpoint["model"]

**Edit for your project.** `load_eval_batch()` returns the inputs and targets every model sees. `measure(model, inputs, targets)` returns the new metrics as a dict. This pair computes loss and gradient norm on the first 32 MNIST test images. Gradients aren't in `state_dict()`, so `measure` computes them anew.

Set `DATA_ROOT` to your evaluation data, relative to this notebook. The example uses the MNIST data downloaded by `examples/v4/train.py`.

In [ ]:
from pathlib import Path

from torchvision import datasets, transforms

DATA_ROOT = Path("../examples/data")


def load_eval_batch():
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"{DATA_ROOT.resolve()} not found. Set DATA_ROOT to your evaluation data.")
    dataset = datasets.MNIST(root=str(DATA_ROOT), train=False, download=False, transform=transforms.ToTensor())
    inputs, targets = next(iter(torch.utils.data.DataLoader(dataset, batch_size=32)))
    return inputs.reshape(-1, 784), targets


loss_fn = nn.CrossEntropyLoss()


def measure(model, inputs, targets):
    model.eval()
    loss = loss_fn(model(inputs), targets)
    loss.backward()
    grad_norm = sum(
        p.grad.double().square().sum() for p in model.parameters() if p.grad is not None
    ).sqrt().item()
    return {"comparison_loss": loss.item(), "grad_norm": grad_norm}

Run the comparison. Each model loads its own saved weights and is measured on the same evaluation batch. The resulting table joins the new metrics to the run's recorded arguments and metrics.

In [ ]:
inputs, targets = load_eval_batch()

measurements = []
for _, row in runs.iterrows():
    checkpoint = flor.load_checkpoint(row.tstamp, CHECKPOINT)
    model = make_model(row)
    try:
        model.load_state_dict(state_dict_of(checkpoint))
    except (KeyError, RuntimeError) as e:
        raise RuntimeError(
            f"Run {row.tstamp}'s {CHECKPOINT} doesn't fit make_model. Edit make_model or state_dict_of."
        ) from e
    measurements.append({"tstamp": row.tstamp, **measure(model, inputs, targets)})

comparison = runs.merge(pd.DataFrame(measurements), on="tstamp", validate="one_to_one")
comparison

The comparison stays in this notebook's dataframe. Loading checkpoints doesn't start a FlorDB run or write replay rows. The metrics are new measurements on the evaluation batch, not values recovered from the original training steps.